# Latent-dimension sweep

Same design as the β sweep: short runs on fold 0 of the same file subset, one per latent size `ncode`. β, the error floor, gradient clipping and batch mixing are fixed at the tuned values, so the only difference between runs is the latent size.

The latent size involves a trade-off:
- **Too small:** normal spectra reconstruct poorly, and finer spectral detail is lost from μ.
- **Too large:** the model can reconstruct anomalies too, which weakens the residual score, and kNN gets less reliable in more dimensions.

So the choice is made on **transient recall at fixed FPR**, with reconstruction quality and latent usage as supporting diagnostics. For each run the notebook records:
- **Recall** of known transients at FPR 10⁻² and 10⁻³ for `rchi2`, `rchi2_snr` (error-floor corrected) and `knn` (latent kNN distance), with ±1σ uncertainties.
- **Validation reduced χ²** (with the error floor) and **validation KL**.
- **KL per latent dimension:** information per dimension in nats. Dimensions near 0 are unused.
- **Effective dimensionality:** how many principal components of standardized μ explain 90%, 99% and 99.9% of its variance.
- **Class separation:** kNN balanced accuracy of SPECTYPE on held-out μ.

In [1]:
from pathlib import Path
import importlib, json, sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import balanced_accuracy_score
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, "/pscratch/sd/z/zixinq")
import vae_model as vae_lib
vae_lib = importlib.reload(vae_lib)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_CKPT = Path("/pscratch/sd/z/zixinq/vae_v2_run1/best.pt")   # configs only
SWEEP_DIR = Path("/pscratch/sd/z/zixinq/vae_ncode_sweep")

NCODES = [8, 16, 24, 32, 48, 64, 128]

# Fixed at the tuned values (same as the beta sweep and vae_training.ipynb).
BETA = 1
FRAC_FLOOR = 0.02
MAX_GRAD_NORM = 3e7
MIX_BATCHES = 32
FILE_FRACTION = 0.2     
SWEEP_EPOCHS = 8
SEED = 42
FPR_LEVELS = [1e-2, 1e-3]

# Optional: reuse finished runs from the beta sweep, e.g. {16: Path(".../vae_tuning/beta_1e-3")}.
# A run is only reused if its data and training settings match this notebook exactly.
REUSE_RUNS = {}

base = torch.load(BASE_CKPT, map_location="cpu", weights_only=True)
model_config, base_data_config = dict(base["model_config"]), dict(base["data_config"])
del base

sweep_train_config = dict(
    epochs=SWEEP_EPOCHS, learning_rate=3e-4, beta_max=BETA, warmup_epochs=2, patience=3,
    weight_power=1.0, max_grad_norm=MAX_GRAD_NORM, log_every=2000, checkpoint_every_batches=2000,
    frac_floor=FRAC_FLOOR,
)

files = list(vae_lib.resolve_hdf5_files(base_data_config["path"]))
rng = np.random.default_rng(SEED)   # same draw as the beta sweep -> same subset
subset = sorted(rng.choice(files, max(1, int(FILE_FRACTION * len(files))), replace=False).tolist())
sweep_data_config = dict(base_data_config, path=subset, n_folds=2, fold=0, mix_batches=MIX_BATCHES)
print(f"sweep uses {len(subset)} of {len(files)} files")

# New transient catalogue (HDF5), read with the same keys and valid-pixel cut as the training spectra.
SOURCE = Path("/global/cfs/cdirs/desi/users/zixinq/data/transient_catalog/transient_spectra_preprocessed.h5")
vae_lib.inspect_hdf5(SOURCE)                       # lists every array in the file
tr_data = vae_lib.load_spectra_h5(SOURCE, **base_data_config)
X_tr, W_tr, tid_tr, spec_tr = tr_data["X"], tr_data["W"], tr_data["TARGETID"], tr_data["SPECTYPE"]
exclude_ids = tr_data["all_targetids"]             # every transient, including rows that fail the cut
DUPLICATES = len(np.unique(tid_tr)) < len(tid_tr)  # several spectra per object -> also report per-object recall
print("compared with a training file:")
vae_lib.compare_to_training(tr_data, files[0], **base_data_config)

sweep uses 337 of 1688 files
M (712, 1000) uint8
TYPE (712,) object
W (712, 1000) float32
X (712, 1000) float32
metadata_json (712,) object
native_valid_frac (712,) float32
norm (712,) float64
offset (712,) float64
raw_npz_path (712,) object
raw_npz_sha256 (712,) object
record_id (712,) object
snr_orig (712,) float64
source_path (712,) object
source_row (712,) int64
status (712,) uint8
targetid (712,) int64
valid_frac (712,) float32
wave_grid (1000,) float64
transient_spectra_preprocessed.h5: 712 spectra, 712 with >= 50 valid pixels, 672 unique TARGETIDs
  keys used: {'flux': 'X', 'weight': 'W', 'targetid': 'targetid'}
  SPECTYPE counts: {'UNKNOWN': 712}
compared with a training file:
  wavelength grid identical to training: True
  median |X|: transients 0.4545 | training 0.4735 | ratio 0.96
  median W: transients 3011 | training 2027 | ratio 1.49
  median valid pixels: transients 1000 | training 1000 | ratio 1.00


In [2]:
def sweep_loader(split, shuffle=False):
    ds = vae_lib.H5SpectralBatches(**sweep_data_config, split=split, shuffle=shuffle,
                                   exclude_targetids=exclude_ids)
    return vae_lib.make_h5_loader(ds, device)


def load_checkpoint(path):
    ckpt = torch.load(path, map_location="cpu", weights_only=True)
    model = vae_lib.VAE(**ckpt["model_config"]).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model, ckpt


@torch.inference_mode()
def score_arrays(model, X, W, batch=1024):
    model.eval()
    out = {"mu": [], "rchi2": [], "snr": []}
    for s in range(0, len(X), batch):
        x, w = vae_lib.masked_inputs(torch.as_tensor(X[s:s + batch], device=device),
                                     torch.as_tensor(W[s:s + batch], device=device))
        recon, mu, _, _ = model(x, sample=False)
        r = vae_lib._tensor_spectra_2d(recon) - x
        valid = w > 0
        out["rchi2"].append(((w * r.square()).sum(1) / valid.sum(1).clamp_min(1)).cpu())
        out["snr"].append(torch.where(valid, x * w.sqrt(), torch.nan).nanmedian(1).values.cpu())
        out["mu"].append(mu.float().cpu())
    return {k: torch.cat(v).numpy() for k, v in out.items()}


def fit_eps(snr, rchi2, nbins=20):
    ok = np.isfinite(snr) & (snr > 0)
    edges = np.quantile(snr[ok], np.linspace(0, 1, nbins + 1))
    b = np.clip(np.searchsorted(edges, snr[ok], side="right") - 1, 0, nbins - 1)
    s = np.array([np.median(snr[ok][b == i]) for i in range(nbins)])
    m = np.array([np.median(rchi2[ok][b == i]) for i in range(nbins)])
    return float(np.sqrt(max(np.sum((m - 1) * s**2) / np.sum(s**4), 0)))


def fpr_of(x, population):
    pop = np.sort(population)
    return (1 + len(pop) - np.searchsorted(pop, x, side="left")) / (len(pop) + 1)


SETTING_KEYS = ("beta_max", "frac_floor", "max_grad_norm", "learning_rate", "warmup_epochs", "epochs")


def reusable(run_dir, ncode):
    # True if a finished run was trained with exactly this notebook's data and settings.
    if not ((run_dir / "best.pt").is_file() and (run_dir / "sweep_scores.npz").is_file()):
        return False, "missing best.pt or sweep_scores.npz"
    ckpt = torch.load(run_dir / "best.pt", map_location="cpu", weights_only=True)
    problems = []
    if ckpt["model_config"].get("ncode") != ncode:
        problems.append(f"ncode {ckpt['model_config'].get('ncode')}")
    for key in SETTING_KEYS:
        if ckpt["train_config"].get(key) != sweep_train_config[key]:
            problems.append(f"{key} {ckpt['train_config'].get(key)} != {sweep_train_config[key]}")
    saved = dict(ckpt["data_config"])
    if list(saved.get("path", [])) != list(sweep_data_config["path"]):
        problems.append("different file subset")
    for key in ("n_folds", "fold", "mix_batches", "val_fraction", "seed"):
        if saved.get(key) != sweep_data_config.get(key):
            problems.append(f"{key} {saved.get(key)} != {sweep_data_config.get(key)}")
    return not problems, "; ".join(problems) or "settings match"


def transients_in_training():
    # Transient TARGETIDs that fall in the train/val split of this subset. A reused run
    # was trained before this catalogue existed, so it may have trained on them.
    import h5py
    key = base_data_config.get("targetid_key", "TARGETID")
    n = 0
    for p in subset:
        with h5py.File(p, "r") as f:
            t = np.asarray(vae_lib.h5_field(f, key)[:]).reshape(-1)
        codes = vae_lib.split_codes(t, sweep_data_config.get("seed", 42),
                                    sweep_data_config.get("val_fraction", 0.1), 2, 0)
        n += int(np.isin(t[codes < 2], exclude_ids).sum())
    return n


n_overlap = transients_in_training() if REUSE_RUNS else 0
if n_overlap:
    print(f"{n_overlap} transient spectra are in the train/val split of this subset: "
          "runs trained before this catalogue may have learned them, so none are reused.")

run_dirs = {}
for ncode in NCODES:
    if ncode in REUSE_RUNS and not n_overlap:
        ok, why = reusable(Path(REUSE_RUNS[ncode]), ncode)
        print(f"ncode={ncode}: reuse {REUSE_RUNS[ncode]}? {ok} ({why})")
        if ok:
            run_dirs[ncode] = Path(REUSE_RUNS[ncode])
            continue
    run_dirs[ncode] = SWEEP_DIR / f"ncode_{ncode}"
print({k: str(v) for k, v in run_dirs.items()})

{8: '/pscratch/sd/z/zixinq/vae_ncode_sweep/ncode_8', 16: '/pscratch/sd/z/zixinq/vae_ncode_sweep/ncode_16', 24: '/pscratch/sd/z/zixinq/vae_ncode_sweep/ncode_24', 32: '/pscratch/sd/z/zixinq/vae_ncode_sweep/ncode_32', 48: '/pscratch/sd/z/zixinq/vae_ncode_sweep/ncode_48', 64: '/pscratch/sd/z/zixinq/vae_ncode_sweep/ncode_64', 128: '/pscratch/sd/z/zixinq/vae_ncode_sweep/ncode_128'}


## Train

Each run takes about as long as one β-sweep run (roughly 2 hours for 8 epochs on this subset). Rerunning the cell resumes an interrupted run and skips finished ones. Every run starts from the same seed, but the weights differ anyway because the layer sizes differ.

In [ ]:
for ncode, out_dir in run_dirs.items():
    if (out_dir / "sweep_scores.npz").is_file():
        print(f"ncode={ncode}: done, skipping")
        continue
    print(f"\n===== ncode = {ncode} =====")
    latest = out_dir / "latest.pt"
    vae_lib.seed_everything(SEED)
    model = vae_lib.VAE(**dict(model_config, ncode=ncode)).to(device)
    vae_lib.fit_vae(model, sweep_loader("train", shuffle=True), sweep_loader("val"), device,
                    out_dir=out_dir, data_config=sweep_data_config,
                    resume_from=latest if latest.is_file() else None, **sweep_train_config)
    hold = vae_lib.score_spectra(model, sweep_loader("holdout"), device)
    val = vae_lib.score_spectra(model, sweep_loader("val"), device)
    np.savez(out_dir / "sweep_scores.npz",
             **{f"hold_{k}": v for k, v in hold.items()}, val_mu=val["mu"])

## Evaluate

The transients are scored here from each run's `best.pt`, so every run, including reused ones, is evaluated on the current catalogue. The KL per dimension uses the first `KL_BATCHES` holdout batches; the holdout is read in order, so every run sees the same spectra. If the catalogue has several spectra per object, recall is also reported per object (`_obj` columns): an object counts as found if any of its spectra is.

In [ ]:
KL_BATCHES = 200
N_REFERENCE = 20_000
CLASSES = ["GALAXY", "QSO", "STAR"]
n_tr = len(X_tr)


@torch.inference_mode()
def kl_per_dim(model, n_batches=KL_BATCHES):
    kl = []
    for b, (x, w, _, _) in enumerate(sweep_loader("holdout")):
        if b == n_batches:
            break
        x, _ = vae_lib.masked_inputs(x.to(device), w.to(device))
        mu, lv = model.encode(x)
        kl.append((0.5 * (mu**2 + lv.exp() - 1 - lv)).mean(0).cpu())
    return torch.stack(kl).mean(0).numpy()


rows, kl_dims, pca_curves = [], {}, {}
for ncode, out_dir in run_dirs.items():
    if not (out_dir / "sweep_scores.npz").is_file():
        print(f"ncode={ncode}: not finished, skipping")
        continue
    d = dict(np.load(out_dir / "sweep_scores.npz"))
    hist = json.loads((out_dir / "history.json").read_text())
    best = min(hist, key=lambda h: h["val_loss"])
    pop = ~np.isin(d["hold_TARGETID"], exclude_ids)
    model, _ = load_checkpoint(out_dir / "best.pt")
    tr = score_arrays(model, X_tr, W_tr)
    kl = kl_per_dim(model)
    del model
    kl_dims[ncode] = kl

    eps = fit_eps(d["hold_snr"][pop], d["hold_rchi2"][pop])
    corr = lambda r, s: r / (1 + (eps * np.nan_to_num(s)) ** 2)
    ref = d["val_mu"][np.random.default_rng(SEED).choice(len(d["val_mu"]), min(N_REFERENCE, len(d["val_mu"])), replace=False)]
    scaler = StandardScaler().fit(ref)
    knn = NearestNeighbors(n_neighbors=35, n_jobs=8).fit(scaler.transform(ref))
    knn_score = lambda mu: knn.kneighbors(scaler.transform(mu))[0].mean(1)
    scores = {
        "rchi2": (d["hold_rchi2"][pop], tr["rchi2"]),
        "rchi2_snr": (corr(d["hold_rchi2"][pop], d["hold_snr"][pop]), corr(tr["rchi2"], tr["snr"])),
        "knn": (knn_score(d["hold_mu"][pop]), knn_score(tr["mu"])),
    }

    mu_std = StandardScaler().fit_transform(d["hold_mu"][pop][:200_000])
    cum = np.cumsum(PCA().fit(mu_std).explained_variance_ratio_)
    pca_curves[ncode] = cum

    lab = np.char.upper(np.char.strip(d["hold_SPECTYPE"].astype(str)))
    sel = np.flatnonzero(pop & np.isin(lab, CLASSES))
    sel = np.random.default_rng(SEED).choice(sel, min(100_000, len(sel)), replace=False)
    m = StandardScaler().fit_transform(d["hold_mu"][sel])
    h = len(sel) // 2
    sep = balanced_accuracy_score(lab[sel][h:], KNeighborsClassifier(15).fit(m[:h], lab[sel][:h]).predict(m[h:]))

    row = dict(ncode=ncode, best_epoch=best["epoch"], val_rchi2=best["val_rchi2"], val_kl=best["val_kl"],
               kl_min=float(kl.min()), dims_under_0p1nat=int((kl < 0.1).sum()),
               pca90=int(np.searchsorted(cum, 0.90) + 1), pca99=int(np.searchsorted(cum, 0.99) + 1),
               pca999=int(np.searchsorted(cum, 0.999) + 1), class_sep=sep, eps=eps,
               clip_frac=best.get("train_clip_frac"))
    for name, (population, positives) in scores.items():
        f = fpr_of(positives, population)
        for a in FPR_LEVELS:
            p = float(np.mean(f <= a))
            row[f"{name}@{a:g}"] = p
            row[f"{name}@{a:g}_err"] = float(np.sqrt(p * (1 - p) / n_tr))
            if DUPLICATES:
                row[f"{name}@{a:g}_obj"] = float(pd.Series(f <= a).groupby(tid_tr).any().mean())
    rows.append(row)

table = pd.DataFrame(rows).set_index("ncode")
show = [c for c in table.columns if not c.endswith("_err")]
display(table[show].round(4))
print(f"recall uncertainties (±1σ) use {n_tr} transient spectra; see the *_err columns")

In [ ]:
nc = table.index.to_numpy()
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for ax, a in zip(axes[0], FPR_LEVELS):
    for i, name in enumerate(["rchi2", "rchi2_snr", "knn"]):
        ax.errorbar(nc + (i - 1) * 0.3, table[f"{name}@{a:g}"], yerr=table[f"{name}@{a:g}_err"],
                    marker="o", capsize=3, label=name)
    ax.axhline(a, color="k", ls=":", lw=1, label="random")
    ax.set(xlabel="ncode", ylabel="transient recall", title=f"Recall at FPR {a:g}", xticks=nc)
    ax.legend(fontsize=8)

ax = axes[1, 0]
ax.plot(nc, table["val_rchi2"], "o-", label="val reduced χ² (with floor)")
ax.set(xlabel="ncode", ylabel="val reduced χ²", xticks=nc, title="Reconstruction")
ax2 = ax.twinx()
ax2.plot(nc, table["class_sep"], "s--", color="C3", label="class separation")
ax2.set_ylabel("kNN balanced accuracy", color="C3")

ax = axes[1, 1]
for ncode, kl in kl_dims.items():
    ax.plot(np.arange(1, len(kl) + 1), np.sort(kl)[::-1], "o-", ms=3, label=f"ncode={ncode}")
ax.axhline(0.1, color="k", ls=":", lw=1)
ax.set(xlabel="latent dimension (sorted by KL)", ylabel="KL per dimension (nats)", yscale="log",
       title="Information per latent dimension")
ax.legend(fontsize=8)
fig.savefig(SWEEP_DIR / "ncode_sweep.png", dpi=130, bbox_inches="tight")
plt.show()

plt.figure(figsize=(6, 4))
for ncode, cum in pca_curves.items():
    plt.plot(np.arange(1, len(cum) + 1), cum, "o-", ms=3, label=f"ncode={ncode}")
for q in (0.9, 0.99):
    plt.axhline(q, color="k", ls=":", lw=0.8)
plt.xlabel("number of principal components of standardized μ"); plt.ylabel("cumulative explained variance")
plt.ylim(0.5, 1.001); plt.legend(fontsize=8); plt.title("Effective dimensionality")
plt.show()

## Reconstructions compared across latent sizes

Example held-out spectra and transients reconstructed by every run. Left: flux (grey) with each run's reconstruction. Right: each run's residual / σ, offset vertically by 8 per run, with its reduced χ² in the legend.

In [ ]:
N_PER_CLASS, N_TRANSIENTS = 5, 3
found = {c: [] for c in CLASSES}
for b, (x, w, lab, tid) in enumerate(sweep_loader("holdout")):
    for i in np.flatnonzero(np.isin(lab, CLASSES) & ~np.isin(tid, exclude_ids)):
        if len(found[lab[i]]) < N_PER_CLASS:
            found[lab[i]].append((x[i].numpy(), w[i].numpy(), f"{lab[i]} {tid[i]}"))
    if all(len(v) == N_PER_CLASS for v in found.values()) or b >= 2000:
        break
examples = [e for c in CLASSES for e in found[c]]
examples += [(X_tr[i], W_tr[i], f"TRANSIENT {tid_tr[i]} ({spec_tr[i]})") for i in range(min(N_TRANSIENTS, len(X_tr)))]
X_ex = np.stack([e[0] for e in examples]).astype(np.float32)
W_ex = np.stack([e[1] for e in examples]).astype(np.float32)
names = [e[2] for e in examples]

import h5py
with h5py.File(subset[0], "r") as f:
    wave = f["wave_grid"][:] if "wave_grid" in f else np.arange(X_ex.shape[1])

recons = {}
with torch.inference_mode():
    for ncode, out_dir in run_dirs.items():
        if not (out_dir / "best.pt").is_file():
            continue
        model, _ = load_checkpoint(out_dir / "best.pt")
        x, w = vae_lib.masked_inputs(torch.as_tensor(X_ex, device=device), torch.as_tensor(W_ex, device=device))
        recons[ncode] = vae_lib._tensor_spectra_2d(model(x, sample=False)[0]).cpu().numpy()
        del model
x, w = X_ex, np.where(np.isfinite(W_ex) & (W_ex > 0), W_ex, 0)

n = len(names)
fig, axes = plt.subplots(n, 2, figsize=(15, 2.3 * n), sharex=True, squeeze=False,
                         gridspec_kw=dict(width_ratios=[3, 2]))
for i in range(n):
    good = w[i] > 0
    axes[i, 0].plot(wave, np.where(good, x[i], np.nan), color="0.6", lw=0.5)
    axes[i, 0].text(0.01, 0.85, names[i], transform=axes[i, 0].transAxes, fontsize=8)
    for k, (ncode, rec) in enumerate(recons.items()):
        pull = (x[i] - rec[i]) * np.sqrt(w[i])
        r = float(np.mean(pull[good] ** 2))
        axes[i, 0].plot(wave, np.where(good, rec[i], np.nan), color=f"C{k}", lw=0.8)
        axes[i, 1].plot(wave, np.where(good, pull + 8 * k, np.nan), color=f"C{k}", lw=0.4,
                        label=f"ncode={ncode}: rχ²={r:.2f}")
    axes[i, 1].legend(fontsize=6, loc="upper right", ncol=2)
axes[0, 0].set_title("flux (grey) and reconstructions", fontsize=10)
axes[0, 1].set_title("residual / σ (offset by 8 per run)", fontsize=10)
fig.tight_layout()
fig.savefig(SWEEP_DIR / "ncode_reconstructions.png", dpi=130, bbox_inches="tight")
plt.show()

## Choosing the latent size

1. **Main criterion: transient recall at FPR 10⁻³** for your main score (`knn`, and `rchi2_snr` if you combine them). Differences within the ±1σ bars are noise.
2. **If two sizes tie, prefer the smaller one.** It's cheaper, kNN works better in fewer dimensions, and it is less able to reconstruct anomalies.
3. **Supporting checks:**
   - `val_rchi2` should fall as `ncode` grows. If it stops improving, the extra dimensions add little.
   - `dims_under_0p1nat > 0` means the model left some dimensions unused, a sign the latent space is larger than it needs to be. At β = 10⁻³ the KL penalty is weak, so dimensions rarely switch off completely. A sharp drop in the sorted-KL curve is the more useful sign.
   - `pca99` well below `ncode` means μ lives in a lower-dimensional subspace.
4. All runs stop at epoch 8. If `best_epoch` equals 8 for the larger models, they were still improving. Confirm the winner and its nearest rival with longer runs before changing `NCODE` in `vae_training.ipynb`.